# WiDS Wildfire Survival Multichannel Blend

In [1]:

import os
import gc
import re
import glob
import json
import math
import random
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.optimize import minimize

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingClassifier

warnings.filterwarnings("ignore")

SEEDS = [11, 29, 47]
N_SPLITS = 5
EPS = 1e-6
LEGACY_BLEND_WEIGHT = 0.06
HORIZONS = [12, 24, 48, 72]

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except Exception:
    HAS_CAT = False

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def locate_dataset_dir() -> str:
    candidate_dirs = []
    roots = ["/kaggle/input", "/kaggle/working", "."]
    for root in roots:
        for p in glob.glob(os.path.join(root, "**", "train.csv"), recursive=True):
            d = os.path.dirname(p)
            needed = ["train.csv", "test.csv", "sample_submission.csv"]
            if all(os.path.exists(os.path.join(d, f)) for f in needed):
                candidate_dirs.append(d)
    if not candidate_dirs:
        raise FileNotFoundError("Could not locate train.csv / test.csv / sample_submission.csv")
    def score_dir(d: str):
        name = d.lower()
        bonus = 0
        if "globaldathon" in name or "wids" in name:
            bonus -= 10
        if "competition" in name:
            bonus -= 5
        return (bonus, len(d), d)
    return sorted(set(candidate_dirs), key=score_dir)[0]


def find_optional_files(pattern: str):
    roots = ["/kaggle/input", "/kaggle/working", "."]
    out = []
    for root in roots:
        out.extend(glob.glob(os.path.join(root, "**", pattern), recursive=True))
    return sorted(set(out))


def safe_clip(x):
    return np.clip(np.asarray(x, dtype=float), EPS, 1.0 - EPS)


def logit(p):
    p = safe_clip(p)
    return np.log(p / (1.0 - p))


def sigmoid(x):
    x = np.asarray(x, dtype=float)
    return 1.0 / (1.0 + np.exp(-x))


def cdf_transform(train_scores, scores):
    train_scores = np.asarray(train_scores, dtype=float)
    scores = np.asarray(scores, dtype=float)
    train_scores = train_scores[np.isfinite(train_scores)]
    if train_scores.size == 0:
        return np.full(scores.shape[0], 0.5, dtype=float)
    ref = np.sort(train_scores)
    return np.searchsorted(ref, scores, side="right") / float(ref.size)


def rank_average_score(signal_dict, time, event):
    names = list(signal_dict.keys())
    cidx = {}
    rank_oof = []
    rank_test = []
    train_refs = {}
    for name, bundle in signal_dict.items():
        train_sig = np.asarray(bundle["oof"], dtype=float)
        test_sig = np.asarray(bundle["test"], dtype=float)
        train_refs[name] = train_sig.copy()
        cidx[name] = concordance_index_harrell(time, event, train_sig)
    raw_w = np.array([max(cidx[n] - 0.5, 1e-6) ** 2 for n in names], dtype=float)
    raw_w = raw_w / raw_w.sum()
    for name in names:
        rank_oof.append(cdf_transform(train_refs[name], signal_dict[name]["oof"]))
        rank_test.append(cdf_transform(train_refs[name], signal_dict[name]["test"]))
    rank_oof = np.column_stack(rank_oof)
    rank_test = np.column_stack(rank_test)
    return rank_oof @ raw_w, rank_test @ raw_w, dict(zip(names, raw_w.tolist())), cidx


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    for c in x.columns:
        if c == "event_id":
            continue
        x[c] = pd.to_numeric(x[c], errors="coerce")

    x["dist_km"] = x["dist_min_ci_0_5h"] / 1000.0
    x["log_dist"] = np.log1p(np.maximum(x["dist_min_ci_0_5h"], 0.0))
    x["close_gap_m"] = np.maximum(x["dist_min_ci_0_5h"] - 5000.0, 0.0)

    x["closing_pos"] = np.maximum(x["closing_speed_m_per_h"], 0.0)
    x["closing_neg"] = np.maximum(-x["closing_speed_m_per_h"], 0.0)
    x["along_pos"] = np.maximum(x["along_track_speed"], 0.0)
    x["advance_pos"] = np.maximum(x["projected_advance_m"], 0.0)
    x["radial_pos"] = np.maximum(x["radial_growth_rate_m_per_h"], 0.0)
    x["growth_rate_pos"] = np.maximum(x["area_growth_rate_ha_per_h"], 0.0)

    effective_close = (
        x["closing_pos"]
        + 0.70 * x["along_pos"]
        + 0.35 * x["radial_pos"] * np.maximum(x["alignment_abs"], 0.0)
        + 1.0
    )
    x["eta_proxy_h"] = x["close_gap_m"] / effective_close
    x["advance_ratio"] = x["advance_pos"] / (x["dist_min_ci_0_5h"] + 100.0)
    x["radial_to_dist"] = np.maximum(x["radial_growth_m"], 0.0) / (x["dist_min_ci_0_5h"] + 100.0)
    x["disp_to_dist"] = x["centroid_displacement_m"] / (x["dist_min_ci_0_5h"] + 100.0)
    x["growth_to_dist"] = x["growth_rate_pos"] / (x["dist_km"] + 0.25)
    x["close_x_align"] = x["closing_pos"] * x["alignment_abs"]
    x["advance_x_align"] = x["advance_pos"] * x["alignment_abs"]
    x["aligned_advance"] = x["alignment_cos"] * x["closing_speed_m_per_h"]
    x["quality_x_close"] = (1.0 - x["low_temporal_resolution_0_5h"]) * x["closing_pos"]
    x["threat_score"] = (
        2.50 / (1.0 + x["dist_km"])
        + 0.0025 * x["close_x_align"]
        + 0.0015 * x["advance_pos"]
        + 0.20 * x["log1p_growth"]
        + 0.12 * x["log1p_area_first"]
        + 0.04 * x["alignment_abs"]
    )
    x["high_risk_flag"] = (
        (x["dist_min_ci_0_5h"] <= 10000.0)
        & (x["closing_pos"] > 0.0)
        & (x["alignment_abs"] >= 0.5)
    ).astype(int)
    x["near_5km"] = (x["dist_min_ci_0_5h"] <= 5000.0).astype(int)
    x["near_10km"] = (x["dist_min_ci_0_5h"] <= 10000.0).astype(int)

    x["hour_sin"] = np.sin(2.0 * np.pi * x["event_start_hour"] / 24.0)
    x["hour_cos"] = np.cos(2.0 * np.pi * x["event_start_hour"] / 24.0)
    x["dow_sin"] = np.sin(2.0 * np.pi * x["event_start_dayofweek"] / 7.0)
    x["dow_cos"] = np.cos(2.0 * np.pi * x["event_start_dayofweek"] / 7.0)
    x["month_sin"] = np.sin(2.0 * np.pi * (x["event_start_month"] - 1.0) / 12.0)
    x["month_cos"] = np.cos(2.0 * np.pi * (x["event_start_month"] - 1.0) / 12.0)

    x.replace([np.inf, -np.inf], np.nan, inplace=True)
    return x


def build_strata(train: pd.DataFrame) -> np.ndarray:
    t = train["time_to_hit_hours"].values
    e = train["event"].values.astype(int)
    s = np.zeros(len(train), dtype=int)
    s[(e == 1) & (t <= 12)] = 0
    s[(e == 1) & (t > 12) & (t <= 24)] = 1
    s[(e == 1) & (t > 24) & (t <= 48)] = 2
    s[(e == 1) & (t > 48)] = 3
    s[(e == 0) & (t < 24)] = 4
    s[(e == 0) & (t >= 24) & (t < 48)] = 5
    s[(e == 0) & (t >= 48)] = 6
    vc = pd.Series(s).value_counts()
    rare = set(vc[vc < N_SPLITS].index.tolist())
    if rare:
        s = np.where(np.isin(s, list(rare)), 100 + e, s)
    return s


def km_censor_fit(time, event):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    censor = 1 - event
    uniq_times = np.sort(np.unique(time))
    surv = 1.0
    surv_times = []
    surv_vals = []
    for t in uniq_times:
        at_risk = np.sum(time >= t)
        d = np.sum((time == t) & (censor == 1))
        if at_risk > 0:
            surv *= (1.0 - d / at_risk)
        surv_times.append(t)
        surv_vals.append(max(surv, EPS))
    return np.asarray(surv_times, dtype=float), np.asarray(surv_vals, dtype=float)


def km_censor_predict(km, query):
    surv_times, surv_vals = km
    query = np.asarray(query, dtype=float)
    idx = np.searchsorted(surv_times, query, side="right") - 1
    out = np.ones_like(query, dtype=float)
    mask = idx >= 0
    out[mask] = surv_vals[idx[mask]]
    return np.clip(out, 0.05, 1.0)


def binary_horizon_target(time, event, horizon, km=None):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    valid = ~((event == 0) & (time < horizon))
    y = ((event == 1) & (time <= horizon)).astype(int)

    if km is None:
        w = np.ones_like(time, dtype=float)
        return valid, y, w

    g_h = km_censor_predict(km, np.full(time.shape[0], horizon))
    g_t = km_censor_predict(km, time)
    w = np.where((event == 1) & (time <= horizon), 1.0 / g_t, 1.0 / g_h)
    w[~valid] = 0.0
    if np.any(valid):
        w_valid = w[valid]
        w[valid] = w_valid / max(np.mean(w_valid), EPS)
    return valid, y, w


def concordance_index_harrell(time, event, risk):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    risk = np.asarray(risk, dtype=float)
    n = len(time)
    concordant = 0.0
    comparable = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            ti, tj = time[i], time[j]
            ei, ej = event[i], event[j]
            ri, rj = risk[i], risk[j]

            if (ei == 1) and (ti < tj):
                comparable += 1.0
                if ri > rj:
                    concordant += 1.0
                elif ri == rj:
                    concordant += 0.5
            elif (ej == 1) and (tj < ti):
                comparable += 1.0
                if rj > ri:
                    concordant += 1.0
                elif ri == rj:
                    concordant += 0.5
    if comparable == 0:
        return 0.5
    return concordant / comparable


def brier_at_horizon(time, event, prob, horizon):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    prob = np.asarray(prob, dtype=float)
    valid = ~((event == 0) & (time < horizon))
    if valid.sum() == 0:
        return 0.25
    y = ((event == 1) & (time <= horizon)).astype(float)
    return np.mean((prob[valid] - y[valid]) ** 2)


def hybrid_metric(time, event, p12, p24, p48, p72):
    cidx = concordance_index_harrell(time, event, p12)
    b24 = brier_at_horizon(time, event, p24, 24)
    b48 = brier_at_horizon(time, event, p48, 48)
    b72 = brier_at_horizon(time, event, p72, 72)
    wbs = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * cidx + 0.7 * (1.0 - wbs), cidx, wbs, b24, b48, b72


def fit_platt(raw_train, y_train, raw_new):
    raw_train = np.asarray(raw_train, dtype=float).reshape(-1)
    y_train = np.asarray(y_train, dtype=int).reshape(-1)
    raw_new = np.asarray(raw_new, dtype=float).reshape(-1)

    if raw_train.size == 0:
        fill = 0.5
        return np.full_like(raw_new, fill, dtype=float), {"type": "constant", "value": fill}
    if np.unique(y_train).size < 2:
        fill = float(np.mean(y_train))
        return np.full_like(raw_new, fill, dtype=float), {"type": "constant", "value": fill}

    X_train = np.column_stack([raw_train, np.tanh(raw_train)])
    X_new = np.column_stack([raw_new, np.tanh(raw_new)])
    model = LogisticRegression(C=2.0, solver="lbfgs", max_iter=2000)
    model.fit(X_train, y_train)
    return model.predict_proba(X_new)[:, 1], model


def fit_platt_prob(prob_train, y_train, prob_new):
    prob_train = safe_clip(prob_train)
    prob_new = safe_clip(prob_new)
    if np.unique(np.asarray(y_train)).size < 2:
        fill = float(np.mean(y_train))
        return np.full_like(prob_new, fill, dtype=float), {"type": "constant", "value": fill}
    X_train = np.column_stack([prob_train, logit(prob_train)])
    X_new = np.column_stack([prob_new, logit(prob_new)])
    model = LogisticRegression(C=4.0, solver="lbfgs", max_iter=2000)
    model.fit(X_train, y_train)
    return model.predict_proba(X_new)[:, 1], model


def shrink_to_prior(prob, prior, alpha):
    prob = safe_clip(prob)
    prior = float(np.clip(prior, EPS, 1.0 - EPS))
    return sigmoid(alpha * logit(prob) + (1.0 - alpha) * logit(prior))

def optimize_blend(P, y, reg=0.002):
    P = np.asarray(P, dtype=float)
    y = np.asarray(y, dtype=float)
    m = P.shape[1]
    if m == 1:
        return np.array([1.0], dtype=float)

    indiv = np.array([np.mean((P[:, i] - y) ** 2) for i in range(m)], dtype=float)
    base = 1.0 / np.maximum(indiv, 1e-9)
    x0 = base / base.sum()

    def obj(w):
        p = np.clip(P @ w, EPS, 1.0 - EPS)
        return np.mean((p - y) ** 2) + reg * np.sum((w - 1.0 / m) ** 2)

    bounds = [(0.0, 1.0)] * m
    cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    try:
        res = minimize(
            obj,
            x0,
            method="SLSQP",
            bounds=bounds,
            constraints=cons,
            options={"maxiter": 200, "ftol": 1e-9, "disp": False},
        )
        if res.success:
            w = np.clip(res.x, 0.0, 1.0)
            w = w / max(w.sum(), EPS)
            return w
    except Exception:
        pass
    return x0


def constant_preds(y_tr, w_tr, n_val, n_test):
    if y_tr.size == 0:
        p = 0.5
    else:
        if w_tr is None or len(w_tr) != len(y_tr):
            p = float(np.mean(y_tr))
        else:
            p = float(np.average(y_tr, weights=w_tr))
    return np.full(n_val, p, dtype=float), np.full(n_test, p, dtype=float)


def fit_lgb_binary(X_tr, y_tr, w_tr, X_va, y_va, w_va, X_te, seed):
    if (not HAS_LGB) or (np.unique(y_tr).size < 2):
        return constant_preds(y_tr, w_tr, len(X_va), len(X_te))
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=2500,
        learning_rate=0.02,
        num_leaves=15,
        max_depth=-1,
        min_child_samples=8,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_alpha=0.20,
        reg_lambda=1.50,
        random_state=seed,
        n_jobs=-1,
    )
    try:
        callbacks = [lgb.early_stopping(120, verbose=False)]
        model.fit(
            X_tr, y_tr,
            sample_weight=w_tr,
            eval_set=[(X_va, y_va)] if len(X_va) else None,
            eval_sample_weight=[w_va] if len(X_va) else None,
            eval_metric="binary_logloss",
            callbacks=callbacks,
        )
    except Exception:
        model.fit(X_tr, y_tr, sample_weight=w_tr)
    pv = model.predict_proba(X_va)[:, 1] if len(X_va) else np.array([])
    pt = model.predict_proba(X_te)[:, 1]
    return pv, pt


def fit_cat_binary(X_tr, y_tr, w_tr, X_va, y_va, w_va, X_te, seed):
    if (not HAS_CAT) or (np.unique(y_tr).size < 2):
        return constant_preds(y_tr, w_tr, len(X_va), len(X_te))
    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        iterations=3000,
        learning_rate=0.02,
        depth=5,
        l2_leaf_reg=6.0,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
    )
    try:
        model.fit(
            X_tr, y_tr,
            sample_weight=w_tr,
            eval_set=(X_va, y_va) if len(X_va) else None,
            use_best_model=bool(len(X_va)),
        )
    except Exception:
        model.fit(X_tr, y_tr, sample_weight=w_tr)
    pv = model.predict_proba(X_va)[:, 1] if len(X_va) else np.array([])
    pt = model.predict_proba(X_te)[:, 1]
    return pv, pt


def fit_xgb_binary(X_tr, y_tr, w_tr, X_va, y_va, w_va, X_te, seed):
    if (not HAS_XGB) or (np.unique(y_tr).size < 2):
        return constant_preds(y_tr, w_tr, len(X_va), len(X_te))
    pos = max(np.sum(y_tr == 1), 1)
    neg = max(np.sum(y_tr == 0), 1)
    spw = float(np.sqrt(neg / pos))
    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_estimators=2500,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=2.0,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_alpha=0.25,
        reg_lambda=2.00,
        gamma=0.0,
        random_state=seed,
        n_jobs=2,
        scale_pos_weight=spw,
    )
    try:
        model.fit(
            X_tr, y_tr,
            sample_weight=w_tr,
            eval_set=[(X_va, y_va)] if len(X_va) else None,
            verbose=False,
        )
    except Exception:
        model.fit(X_tr, y_tr, sample_weight=w_tr)
    pv = model.predict_proba(X_va)[:, 1] if len(X_va) else np.array([])
    pt = model.predict_proba(X_te)[:, 1]
    return pv, pt


def fit_phys_binary(X_tr, y_tr, w_tr, X_va, X_te):
    if np.unique(y_tr).size < 2:
        return constant_preds(y_tr, w_tr, len(X_va), len(X_te))
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=0.6, solver="lbfgs", max_iter=2000)),
    ])
    try:
        model.fit(X_tr, y_tr, clf__sample_weight=w_tr)
    except Exception:
        try:
            model.fit(X_tr, y_tr, sample_weight=w_tr)
        except Exception:
            model.fit(X_tr, y_tr)
    pv = model.predict_proba(X_va)[:, 1] if len(X_va) else np.array([])
    pt = model.predict_proba(X_te)[:, 1]
    return pv, pt


def fit_fallback_binary(X_tr, y_tr, w_tr, X_va, y_va, X_te, seed):
    if np.unique(y_tr).size < 2:
        return constant_preds(y_tr, w_tr, len(X_va), len(X_te))
    model = HistGradientBoostingClassifier(
        learning_rate=0.03,
        max_depth=4,
        max_iter=400,
        min_samples_leaf=5,
        l2_regularization=1.0,
        random_state=seed,
    )
    model.fit(X_tr, y_tr, sample_weight=w_tr)
    pv = model.predict_proba(X_va)[:, 1] if len(X_va) else np.array([])
    pt = model.predict_proba(X_te)[:, 1]
    return pv, pt


def fit_binary_family(kind, X_tr, y_tr, w_tr, X_va, y_va, w_va, X_te, seed):
    if len(X_tr) == 0:
        return constant_preds(y_tr, w_tr, len(X_va), len(X_te))
    if kind == "lgb":
        if HAS_LGB:
            return fit_lgb_binary(X_tr, y_tr, w_tr, X_va, y_va, w_va, X_te, seed)
        return fit_fallback_binary(X_tr, y_tr, w_tr, X_va, y_va, X_te, seed)
    if kind == "cat":
        if HAS_CAT:
            return fit_cat_binary(X_tr, y_tr, w_tr, X_va, y_va, w_va, X_te, seed)
        return fit_fallback_binary(X_tr, y_tr, w_tr, X_va, y_va, X_te, seed)
    if kind == "xgb":
        if HAS_XGB:
            return fit_xgb_binary(X_tr, y_tr, w_tr, X_va, y_va, w_va, X_te, seed)
        return fit_fallback_binary(X_tr, y_tr, w_tr, X_va, y_va, X_te, seed)
    raise ValueError(kind)


def fit_cox_raw(X_tr, y_signed, X_va, X_te, seed):
    if (not HAS_XGB) or (len(np.unique(np.sign(y_signed))) < 2):
        median_risk = 0.0
        return np.full(len(X_va), median_risk), np.full(len(X_te), median_risk)
    model = xgb.XGBRegressor(
        objective="survival:cox",
        eval_metric="cox-nloglik",
        tree_method="hist",
        n_estimators=1800,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=2.0,
        subsample=0.90,
        colsample_bytree=0.85,
        reg_alpha=0.10,
        reg_lambda=2.0,
        random_state=seed,
        n_jobs=2,
    )
    try:
        model.fit(X_tr, y_signed, eval_set=[(X_va, np.zeros(len(X_va)))], verbose=False)
    except Exception:
        model.fit(X_tr, y_signed)
    return model.predict(X_va), model.predict(X_te)


def fit_event_time_raw(X_tr, y_logt, X_va, X_te, seed):
    if len(y_logt) < 8:
        fill = float(np.median(y_logt)) if len(y_logt) else 1.0
        return np.full(len(X_va), fill), np.full(len(X_te), fill)
    model = GradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.03,
        n_estimators=500,
        max_depth=2,
        subsample=0.85,
        random_state=seed,
    )
    model.fit(X_tr, y_logt)
    return model.predict(X_va), model.predict(X_te)


set_seed(42)

data_dir = locate_dataset_dir()
train = pd.read_csv(os.path.join(data_dir, "train.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))
sample_sub = pd.read_csv(os.path.join(data_dir, "sample_submission.csv"))

train_id = train["event_id"].copy()
test_id = test["event_id"].copy()

all_df = pd.concat(
    [
        train.drop(columns=["time_to_hit_hours", "event"], errors="ignore"),
        test.copy(),
    ],
    axis=0,
    ignore_index=True,
)
all_df = add_features(all_df)
train_fe = all_df.iloc[: len(train)].copy()
test_fe = all_df.iloc[len(train):].copy()

feature_cols = [c for c in train_fe.columns if c != "event_id"]
compact_cols = [
    c for c in [
        "dist_min_ci_0_5h", "dist_km", "log_dist", "close_gap_m", "closing_pos",
        "advance_pos", "alignment_abs", "along_pos", "radial_pos", "log1p_area_first",
        "log1p_growth", "threat_score", "eta_proxy_h", "advance_ratio", "quality_x_close",
        "hour_sin", "hour_cos", "month_sin", "month_cos", "near_5km", "near_10km"
    ]
    if c in train_fe.columns
]

X_train_df = train_fe[feature_cols].copy()
X_test_df = test_fe[feature_cols].copy()

time = train["time_to_hit_hours"].astype(float).values
event = train["event"].astype(int).values
strata = build_strata(train)

dist_thr = float(np.nanmedian(train_fe["dist_min_ci_0_5h"].values))
zone_train = (train_fe["dist_min_ci_0_5h"].values <= dist_thr).astype(int)
zone_test_bin = (test_fe["dist_min_ci_0_5h"].values <= dist_thr).astype(int)
zones = [0, 1]

candidate_names = ["lgb", "xgb", "cat", "phys"]
oof_sum = {name: {h: np.zeros(len(train), dtype=float) for h in HORIZONS} for name in candidate_names}
oof_cnt = {name: {h: np.zeros(len(train), dtype=float) for h in HORIZONS} for name in candidate_names}
test_sum = {name: {h: np.zeros(len(test), dtype=float) for h in HORIZONS} for name in candidate_names}
test_cnt = {name: {h: 0 for h in HORIZONS} for name in candidate_names}

raw_cox_oof_sum = np.zeros(len(train), dtype=float)
raw_cox_oof_cnt = np.zeros(len(train), dtype=float)
raw_cox_test_sum = np.zeros(len(test), dtype=float)
raw_cox_test_cnt = 0

raw_time_oof_sum = np.zeros(len(train), dtype=float)
raw_time_oof_cnt = np.zeros(len(train), dtype=float)
raw_time_test_sum = np.zeros(len(test), dtype=float)
raw_time_test_cnt = 0

for repeat_idx, seed in enumerate(SEEDS):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(train)), strata)):
        Xtr_raw = X_train_df.iloc[tr_idx].copy()
        Xva_raw = X_train_df.iloc[va_idx].copy()
        Xte_raw = X_test_df.copy()

        imp = SimpleImputer(strategy="median")
        Xtr = imp.fit_transform(Xtr_raw)
        Xva = imp.transform(Xva_raw)
        Xte = imp.transform(Xte_raw)

        imp_comp = SimpleImputer(strategy="median")
        Xtr_comp = imp_comp.fit_transform(Xtr_raw[compact_cols])
        Xva_comp = imp_comp.transform(Xva_raw[compact_cols])
        Xte_comp = imp_comp.transform(Xte_raw[compact_cols])

        km = km_censor_fit(time[tr_idx], event[tr_idx])

        signed_y = np.where(event[tr_idx] == 1, time[tr_idx], -time[tr_idx])
        try:
            pv_cox, pt_cox = fit_cox_raw(Xtr, signed_y, Xva, Xte, seed + fold)
            raw_cox_oof_sum[va_idx] += pv_cox
            raw_cox_oof_cnt[va_idx] += 1
            raw_cox_test_sum += pt_cox
            raw_cox_test_cnt += 1
        except Exception:
            pass

        event_mask_tr = event[tr_idx] == 1
        try:
            pv_time, pt_time = fit_event_time_raw(
                Xtr[event_mask_tr],
                np.log1p(time[tr_idx][event_mask_tr]),
                Xva,
                Xte,
                seed + 100 + fold,
            )
            raw_time_oof_sum[va_idx] += pv_time
            raw_time_oof_cnt[va_idx] += 1
            raw_time_test_sum += pt_time
            raw_time_test_cnt += 1
        except Exception:
            pass

        for horizon in HORIZONS:
            valid_tr, y_tr_all, w_tr_all = binary_horizon_target(time[tr_idx], event[tr_idx], horizon, km)
            valid_va, y_va_all, w_va_all = binary_horizon_target(time[va_idx], event[va_idx], horizon, None)

            y_tr = y_tr_all[valid_tr]
            w_tr = w_tr_all[valid_tr]
            y_va = y_va_all[valid_va]
            w_va = w_va_all[valid_va]

            Xtr_h = Xtr[valid_tr]
            Xva_h = Xva[valid_va]
            Xtr_comp_h = Xtr_comp[valid_tr]
            Xva_comp_h = Xva_comp[valid_va]

            for name in ["lgb", "xgb", "cat"]:
                pv, pt = fit_binary_family(
                    name,
                    Xtr_h,
                    y_tr,
                    w_tr,
                    Xva_h,
                    y_va,
                    w_va,
                    Xte,
                    seed + horizon + fold,
                )
                full_pv = np.full(len(va_idx), np.mean(y_tr) if len(y_tr) else 0.5, dtype=float)
                if valid_va.sum():
                    full_pv[valid_va] = pv
                oof_sum[name][horizon][va_idx] += full_pv
                oof_cnt[name][horizon][va_idx] += 1
                test_sum[name][horizon] += pt
                test_cnt[name][horizon] += 1

            pv_phys, pt_phys = fit_phys_binary(
                Xtr_comp_h,
                y_tr,
                w_tr,
                Xva_comp,
                Xte_comp,
            )
            oof_sum["phys"][horizon][va_idx] += pv_phys
            oof_cnt["phys"][horizon][va_idx] += 1
            test_sum["phys"][horizon] += pt_phys
            test_cnt["phys"][horizon] += 1

        gc.collect()

oof = {}
pred = {}
for name in candidate_names:
    oof[name] = {}
    pred[name] = {}
    for horizon in HORIZONS:
        cnt = np.maximum(oof_cnt[name][horizon], 1.0)
        oof[name][horizon] = np.clip(oof_sum[name][horizon] / cnt, EPS, 1.0 - EPS)
        pred[name][horizon] = np.clip(
            test_sum[name][horizon] / max(test_cnt[name][horizon], 1),
            EPS,
            1.0 - EPS,
        )

raw_cox_oof = raw_cox_oof_sum / np.maximum(raw_cox_oof_cnt, 1.0)
raw_cox_test = raw_cox_test_sum / max(raw_cox_test_cnt, 1)
raw_time_oof = raw_time_oof_sum / np.maximum(raw_time_oof_cnt, 1.0)
raw_time_test = raw_time_test_sum / max(raw_time_test_cnt, 1)

blend_report = {
    "data_dir": data_dir,
    "dist_threshold_for_zone_split_m": dist_thr,
    "packages": {"lightgbm": HAS_LGB, "catboost": HAS_CAT, "xgboost": HAS_XGB},
    "zone_blend_weights": {},
    "raw_signal_cindex": {},
    "raw_signal_rank_weights": {},
}

# Calibrate Cox and events-only time model to each horizon
cox_prob_oof, cox_prob_test = {}, {}
time_prob_oof, time_prob_test = {}, {}
for horizon in HORIZONS:
    valid, y, _ = binary_horizon_target(time, event, horizon, None)
    cox_prob_oof[horizon] = np.full(len(train), np.mean(y[valid]) if valid.sum() else 0.5)
    cox_prob_test[horizon] = np.full(len(test), np.mean(y[valid]) if valid.sum() else 0.5)
    time_prob_oof[horizon] = np.full(len(train), np.mean(y[valid]) if valid.sum() else 0.5)
    time_prob_test[horizon] = np.full(len(test), np.mean(y[valid]) if valid.sum() else 0.5)

    cox_fit, _ = fit_platt(raw_cox_oof[valid], y[valid], np.r_[raw_cox_oof, raw_cox_test])
    time_fit, _ = fit_platt(-raw_time_oof[valid], y[valid], np.r_[-raw_time_oof, -raw_time_test])

    cox_prob_oof[horizon] = np.clip(cox_fit[: len(train)], EPS, 1.0 - EPS)
    cox_prob_test[horizon] = np.clip(cox_fit[len(train):], EPS, 1.0 - EPS)
    time_prob_oof[horizon] = np.clip(time_fit[: len(train)], EPS, 1.0 - EPS)
    time_prob_test[horizon] = np.clip(time_fit[len(train):], EPS, 1.0 - EPS)

# 12h: ranking-first blend
signal_dict = {
    "lgb12": {"oof": oof["lgb"][12], "test": pred["lgb"][12]},
    "xgb12": {"oof": oof["xgb"][12], "test": pred["xgb"][12]},
    "cat12": {"oof": oof["cat"][12], "test": pred["cat"][12]},
    "phys12": {"oof": oof["phys"][12], "test": pred["phys"][12]},
    "cox_raw": {"oof": raw_cox_oof, "test": raw_cox_test},
    "time_raw": {"oof": -raw_time_oof, "test": -raw_time_test},
}
p12_rank_oof, p12_rank_test, rank_w, rank_cidx = rank_average_score(signal_dict, time, event)
blend_report["raw_signal_rank_weights"] = rank_w
blend_report["raw_signal_cindex"] = rank_cidx

valid12, y12, _ = binary_horizon_target(time, event, 12, None)
p12_all, _ = fit_platt_prob(p12_rank_oof[valid12], y12[valid12], np.r_[p12_rank_oof, p12_rank_test])
p12_oof = np.clip(p12_all[: len(train)], EPS, 1.0 - EPS)
p12_test = np.clip(p12_all[len(train):], EPS, 1.0 - EPS)

# 24/48/72: calibration-first zone-stratified probability blend
final_oof = {12: p12_oof}
final_test = {12: p12_test}

for horizon in [24, 48, 72]:
    valid, y, _ = binary_horizon_target(time, event, horizon, None)

    P_oof = np.column_stack([
        oof["lgb"][horizon],
        oof["xgb"][horizon],
        oof["cat"][horizon],
        oof["phys"][horizon],
        cox_prob_oof[horizon],
        time_prob_oof[horizon],
    ])
    P_test = np.column_stack([
        pred["lgb"][horizon],
        pred["xgb"][horizon],
        pred["cat"][horizon],
        pred["phys"][horizon],
        cox_prob_test[horizon],
        time_prob_test[horizon],
    ])
    names = ["lgb", "xgb", "cat", "phys", "cox", "time"]

    global_w = optimize_blend(P_oof[valid], y[valid], reg=0.003)
    zone_w = {}
    zone_oof = np.zeros(len(train), dtype=float)
    zone_test_prob = np.zeros(len(test), dtype=float)

    for z in zones:
        mask = valid & (zone_train == z)
        if mask.sum() >= 30 and np.unique(y[mask]).size > 1:
            wz = optimize_blend(P_oof[mask], y[mask], reg=0.005)
        else:
            wz = global_w.copy()
        zone_w[int(z)] = dict(zip(names, wz.tolist()))
        zone_oof[zone_train == z] = np.clip(P_oof[zone_train == z] @ wz, EPS, 1.0 - EPS)
        zone_test_prob[zone_test_bin == z] = np.clip(P_test[zone_test_bin == z] @ wz, EPS, 1.0 - EPS)

    blend_report["zone_blend_weights"][str(horizon)] = zone_w

    calibrated_oof = np.zeros(len(train), dtype=float)
    calibrated_test = np.zeros(len(test), dtype=float)
    global_fit, _ = fit_platt_prob(zone_oof[valid], y[valid], np.r_[zone_oof, zone_test_prob])
    global_oof = np.clip(global_fit[: len(train)], EPS, 1.0 - EPS)
    global_te = np.clip(global_fit[len(train):], EPS, 1.0 - EPS)

    for z in zones:
        mask = valid & (zone_train == z)
        if mask.sum() >= 30 and np.unique(y[mask]).size > 1:
            fit_all, _ = fit_platt_prob(zone_oof[mask], y[mask], np.r_[zone_oof[zone_train == z], zone_test_prob[zone_test_bin == z]])
            calibrated_oof[zone_train == z] = np.clip(fit_all[: np.sum(zone_train == z)], EPS, 1.0 - EPS)
            calibrated_test[zone_test_bin == z] = np.clip(fit_all[np.sum(zone_train == z):], EPS, 1.0 - EPS)
        else:
            calibrated_oof[zone_train == z] = global_oof[zone_train == z]
            calibrated_test[zone_test_bin == z] = global_te[zone_test_bin == z]

    final_oof[horizon] = calibrated_oof
    final_test[horizon] = calibrated_test

# Optional blend with prior public-LB submissions if mounted
legacy_files = find_optional_files("samplecsv_sub*.csv")
legacy_map = {
    1: 0.96617,
    2: 0.96540,
    3: 0.96961,
    4: 0.97252,
    5: 0.97167,
    6: 0.97204,
    7: 0.97252,
    8: 0.97058,
}
legacy_frames = []
legacy_weights = []
sample_key = sample_sub["event_id"].astype(str)

for path in legacy_files:
    m = re.search(r"samplecsv_sub(\d+)\.csv$", os.path.basename(path))
    if m is None:
        continue
    sub_idx = int(m.group(1))
    score = legacy_map.get(sub_idx, None)
    if score is None:
        continue
    try:
        df = pd.read_csv(path)
        if list(df.columns) != ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]:
            continue
        aligned = df.copy()
        aligned["event_id"] = aligned["event_id"].astype(str)
        if set(aligned["event_id"]) != set(sample_key):
            continue
        aligned = aligned.set_index("event_id").loc[sample_key].reset_index(drop=True)
        legacy_frames.append(aligned[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].astype(float).values)
        legacy_weights.append(max(score - 0.964, 1e-6))
    except Exception:
        continue

legacy_used = False
if legacy_frames:
    lw = np.asarray(legacy_weights, dtype=float)
    lw = lw / lw.sum()
    legacy_avg = np.tensordot(lw, np.stack(legacy_frames, axis=0), axes=(0, 0))
    final_test[12] = (1.0 - 0.5 * LEGACY_BLEND_WEIGHT) * final_test[12] + (0.5 * LEGACY_BLEND_WEIGHT) * legacy_avg[:, 0]
    for h, col in zip([24, 48, 72], [1, 2, 3]):
        final_test[h] = (1.0 - LEGACY_BLEND_WEIGHT) * final_test[h] + LEGACY_BLEND_WEIGHT * legacy_avg[:, col]
    legacy_used = True

# Horizon-wise logit shrinkage toward empirical prior to reduce overconfident tails
blend_report["shrinkage"] = {}
for horizon in [12, 24, 48, 72]:
    valid_h, y_h, _ = binary_horizon_target(time, event, horizon, None)
    if np.any(valid_h):
        prior_h = float(np.mean(y_h[valid_h]))
    else:
        prior_h = float(np.mean(event))
    best_alpha = 1.0
    best_loss = np.inf
    for alpha in np.linspace(0.55, 1.00, 10):
        p_try = shrink_to_prior(final_oof[horizon][valid_h], prior_h, float(alpha))
        loss = float(np.mean((p_try - y_h[valid_h]) ** 2)) if np.any(valid_h) else 0.0
        if loss < best_loss:
            best_loss = loss
            best_alpha = float(alpha)
    final_oof[horizon] = shrink_to_prior(final_oof[horizon], prior_h, best_alpha)
    final_test[horizon] = shrink_to_prior(final_test[horizon], prior_h, best_alpha)
    blend_report["shrinkage"][str(horizon)] = {
        "prior": prior_h,
        "alpha": best_alpha,
        "valid_n": int(np.sum(valid_h)),
        "oof_brier_after_shrink": best_loss,
    }

# Row-wise monotonicity
oof_mat = np.column_stack([final_oof[12], final_oof[24], final_oof[48], final_oof[72]])
test_mat = np.column_stack([final_test[12], final_test[24], final_test[48], final_test[72]])

oof_mat = np.maximum.accumulate(np.clip(oof_mat, 0.0, 1.0), axis=1)
test_mat = np.maximum.accumulate(np.clip(test_mat, 0.0, 1.0), axis=1)

local_hybrid, local_cidx, local_wbs, b24, b48, b72 = hybrid_metric(
    time, event, oof_mat[:, 0], oof_mat[:, 1], oof_mat[:, 2], oof_mat[:, 3]
)
blend_report["legacy_submission_blend_used"] = legacy_used
blend_report["legacy_submission_files_found"] = legacy_files
blend_report["local_oof_metrics"] = {
    "hybrid": float(local_hybrid),
    "cindex": float(local_cidx),
    "weighted_brier": float(local_wbs),
    "brier_24h": float(b24),
    "brier_48h": float(b48),
    "brier_72h": float(b72),
}

submission = sample_sub[["event_id"]].copy()
submission["prob_12h"] = test_mat[:, 0]
submission["prob_24h"] = test_mat[:, 1]
submission["prob_48h"] = test_mat[:, 2]
submission["prob_72h"] = test_mat[:, 3]

# Final safety checks
assert list(submission.columns) == ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
assert submission["event_id"].astype(str).tolist() == sample_sub["event_id"].astype(str).tolist()
assert submission["event_id"].astype(str).nunique() == len(submission)
probs = submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].values
assert np.all(np.isfinite(probs))
assert np.all((probs >= -1e-9) & (probs <= 1 + 1e-9))
assert np.all(probs[:, 0] <= probs[:, 1] + 1e-12)
assert np.all(probs[:, 1] <= probs[:, 2] + 1e-12)
assert np.all(probs[:, 2] <= probs[:, 3] + 1e-12)

submission.to_csv("submission.csv", index=False)
with open("blend_report.json", "w") as f:
    json.dump(blend_report, f, indent=2)

oof_out = pd.DataFrame({
    "event_id": train_id,
    "oof_prob_12h": oof_mat[:, 0],
    "oof_prob_24h": oof_mat[:, 1],
    "oof_prob_48h": oof_mat[:, 2],
    "oof_prob_72h": oof_mat[:, 3],
    "time_to_hit_hours": time,
    "event": event,
})
oof_out.to_csv("oof_debug.csv", index=False)

print("data_dir:", data_dir)
print("legacy_files_found:", len(legacy_files), "| legacy_used:", legacy_used)
print("zone distance threshold (m):", round(dist_thr, 3))
print("packages:", {"lightgbm": HAS_LGB, "catboost": HAS_CAT, "xgboost": HAS_XGB})
print("local OOF hybrid:", round(local_hybrid, 6))
print("local OOF c-index:", round(local_cidx, 6))
print("local OOF weighted brier:", round(local_wbs, 6))
print("local OOF brier@24 / @48 / @72:", round(b24, 6), round(b48, 6), round(b72, 6))
print()
print(submission.head())


[LightGBM] [Info] Number of positive: 39, number of negative: 132
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001303 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 799
[LightGBM] [Info] Number of data points in the train set: 171, number of used features: 51
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.224640 -> initscore=-1.238826
[LightGBM] [Info] Start training from score -1.238826
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
